# 01 — The Playwright Crawler

> **AEE Bootcamp | Mini Project 03 — Part 1 [15 Marks]**  
> Scrapes the Kapruka product catalog, enriches it via LLM, and audits the final production dataset.

---

## Overview

This notebook implements **Part 1: Building the Live Data Pipeline**. The Kapruka Gift-Concierge Agent is only as good as the product data it reasons over. This crawler is the foundation of the entire system — it feeds real, live data into the downstream memory and agent layers.

### What this notebook does

| Step | Script | Output |
|---|---|---|
| 1. Scrape | `kapruka_crawler.py` | Raw product records from kapruka.com |
| 2. Enrich | `clean_and_patch.py` | LLM-standardized `catalog.json` |
| 3. Audit | Inline code | Production readiness report |

### Technical Challenges
- **Dynamic loading**: kapruka.com renders content via JavaScript, requiring a headless browser (Playwright) rather than a simple HTTP request.
- **Pagination**: The crawler must follow multi-page product listings to build a complete catalog.
- **Data Quality**: Raw scraped fields (prices, descriptions) are inconsistent and must be normalised via an LLM pass before they can be used for RAG.

---

> ⚠️ **Submission Rule**: No orchestration frameworks (LangGraph/CrewAI). A 50% mark penalty applies. All agent logic must be implemented from scratch.

## Cell 1 — Environment Setup

Establishes the project root and the path to the final `catalog.json` artifact. The path logic handles both running from a `notebooks/` subdirectory and from the project root directly.

In [ ]:
import os
import sys
import subprocess
import json
from pathlib import Path
from collections import Counter

NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
CATALOG_PATH = PROJECT_ROOT / "data" / "catalog.json"

print(f"Project Root: {PROJECT_ROOT}")

## Cell 2 — Launch the Scraper

**Script**: `scraper/kapruka_crawler.py`

This script uses **Playwright** in headless mode to:
1. Navigate through the Kapruka product category pages.
2. Wait for JavaScript-rendered content to load.
3. Extract the following fields for each product:
   - `name` — Product title
   - `price` — Listed price (raw string, to be cleaned in Cell 3)
   - `description` — Product description text
   - `availability` — In-stock / out-of-stock status
   - `product_url` — Canonical product page URL
   - `category` — Mega-category (e.g., Cakes, Flowers, Electronics)
4. Persist the raw output as a JSON file for the enrichment step.

Expect this to take several minutes depending on the number of pages crawled.

In [ ]:
scraper_path = PROJECT_ROOT / "scraper" / "kapruka_crawler.py"

print("🚀 Launching Scraper with Multi-Page Depth...")
subprocess.run([sys.executable, str(scraper_path)], cwd=str(PROJECT_ROOT), check=True)
print("✅ Scrape Finished.")

## Cell 3 — LLM Enrichment & Standardization

**Script**: `scraper/clean_and_patch.py`

Raw scraped data is messy. This step sends each product record through an LLM to:

- **Normalise prices** → Convert all price strings (e.g., `"Rs. 1,250/-"`) to clean **Sri Lankan Rupee integers**.
- **Standardise descriptions** → Rewrite free-form text into a consistent format, and prepend a `SAFETY:` tag that signals allergy/safety content is present (required by the Reflection Loop in Part 4).
- **Strip internal flags** → Remove any `llm_enriched` debug keys before writing the production file.
- **Write `catalog.json`** → The final, clean artifact consumed by the Qdrant ingestion pipeline in Notebook 02.

> This is a **streaming pipeline** — output is printed line-by-line as each batch is processed, so you get live progress without waiting for the entire catalog.

In [ ]:
patcher_path = PROJECT_ROOT / "scraper" / "clean_and_patch.py"

print("🚀 Launching LLM Enrichment & Standardization...")

process = subprocess.Popen(
    [sys.executable, str(patcher_path)],
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding='utf-8',
    errors='replace',
    bufsize=1
)

try:
    for line in iter(process.stdout.readline, ""):
        print(line, end="")
except UnicodeDecodeError:
    print("\n⚠️ Encoding glitch caught, continuing stream...")

process.stdout.close()
process.wait()
print("\n✅ Enrichment Pipeline Finished.")

## Cell 4 — Final Production Data Audit

Before this catalog is ingested into Qdrant (Part 2), it must pass a **Gold Tier** quality gate. This audit verifies five hard requirements:

| Check | Why it Matters |
|---|---|
| ✅ Total product count | Confirms multi-page depth was reached |
| ✅ Sri Lankan integer prices | Required for price-based filtering in the Catalog Agent |
| ✅ URLs preserved | Enables direct product links in agent responses |
| ✅ Dev flags removed | Ensures no internal debug metadata leaks into production |
| ✅ `SAFETY:` tag in descriptions | Required by the Reflection Loop (Part 4) to detect allergen information |

The **Top Mega-Categories** breakdown confirms the crawler reached a broad cross-section of the Kapruka catalog (Cakes, Flowers, Groceries, etc.).

In [ ]:
with open(CATALOG_PATH, "r", encoding="utf-8") as f:
    final_data = json.load(f)

has_urls           = all("product_url" in item for item in final_data)
has_internal_flags = any("llm_enriched" in item for item in final_data)
is_semantic_ready  = all("SAFETY:" in item.get("description", "") for item in final_data)
int_prices         = sum(1 for item in final_data if isinstance(item.get("price"), int))
counts             = Counter(item.get("category") for item in final_data)

print("=" * 50)
print("📦 FINAL PRODUCTION DATA AUDIT (GOLD TIER)")
print("=" * 50)
print(f"Total Products           : {len(final_data):,}")
print(f"Sri Lankan Integer Prices: {int_prices:,} ✅")
print(f"URLs Preserved           : {'YES' if has_urls else 'NO'} ✅")
print(f"Dev Flags Removed        : {'YES' if not has_internal_flags else 'NO'} ✅")
print(f"Safety Ready (Part 4)    : {'YES' if is_semantic_ready else 'NO'} ✅")
print("-" * 50)
print("Top Mega-Categories:")
for cat, count in counts.most_common(5):
    print(f"{cat:<25} : {count}")
print("=" * 50)

## Cell 5 — Identify Items Missing SAFETY Tag

The `SAFETY:` tag is a contract between the Data Pipeline (Part 1) and the Reflection Loop (Part 4). Any product description **without** this tag will fail the allergen-check in the reflection step, potentially allowing an unsafe recommendation to reach the customer.

This cell surfaces all such failures so they can be investigated and re-patched before moving to the next stage.

> **Expected result**: `Total items missing SAFETY tag: 0`  
> Any non-zero count here should be treated as a blocker before proceeding to Notebook 02.

In [ ]:
with open(CATALOG_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

failed_items = [item for item in data if "SAFETY:" not in item.get("description", "")]

print(f"Total items missing SAFETY tag: {len(failed_items)}")
for item in failed_items[:]:
    print(f"ID: {item['id']} | Name: {item['name']}")

---

## ✅ Part 1 Complete

If all audit checks pass and zero items are missing the `SAFETY:` tag, the `catalog.json` artifact is production-ready.

**Next step → [02_cognitive_memory_lab.ipynb](02_cognitive_memory_lab.ipynb)**  
Ingest `catalog.json` into the Qdrant vector store and validate all three tiers of the cognitive memory stack.